In [0]:
dbutils.widgets.text("source_system", "")
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("source_file_format", "")
dbutils.widgets.text("load_type", "")

source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")
load_type = dbutils.widgets.get("load_type")

import re

normalized_load_type = (load_type or "").strip().upper()
normalized_source_file_format = (source_file_format or "").strip().lower()

if normalized_load_type not in {"FULL", "APPEND", "MERGE"}:
    raise ValueError(
        f"Invalid load_type '{load_type}'. Expected FULL, APPEND, or MERGE; "
        "the notebook will not assume FULL."
    )
if normalized_source_file_format not in {"", "csv", "parquet"}:
    raise ValueError(
        f"Unsupported source_file_format '{source_file_format}'. Expected csv or parquet."
    )

# These metadata values become path/table identifiers. Accept simple names only.
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", source_system or ""):
    raise ValueError(f"Invalid source_system '{source_system}'.")
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", table_name or ""):
    raise ValueError(f"Invalid Bronze table_name '{table_name}'.")
if not re.fullmatch(r"[A-Za-z0-9_-]{1,160}", landing_timestamp or ""):
    raise ValueError("landing_timestamp is missing or contains unsupported characters.")


In [0]:
base_landing_path = "/Volumes/healthgrid/landing/landing_volume"
source_path = f"{base_landing_path}/{source_system}"
table_path = f"{source_path}/{table_name}"
landing_path = f"{table_path}/{landing_timestamp}/"

def is_missing_path_error(exc: Exception) -> bool:
    error_text = str(exc).lower()
    return any(token in error_text for token in (
        "filenotfoundexception",
        "pathnotfound",
        "path does not exist",
        "no such file or directory",
    ))

def stop_for_no_data(reason: str) -> None:
    message = (
        f"{reason} source_system={source_system}, table_name={table_name}, "
        f"load_type={normalized_load_type}, landing_timestamp={landing_timestamp}."
    )
    # A FULL CSV source may have no newly eligible file versions because
    # every source file is already PROCESSED in the manifest. This is a valid
    # no-op; ADF separately fails when the source inventory contains no CSVs.
    if normalized_load_type == "FULL" and not (
        source_system == "adls_csv" and reason == "No landing data files found for this batch."
    ):
        raise RuntimeError(message + " FULL loads cannot proceed without an actual data file.")
    print(message + " Returning NO_DATA; Silver and watermark updates must be skipped.")
    dbutils.notebook.exit("NO_DATA")

# Verify the configured landing volume itself. A missing volume or access error
# is a deployment problem, not an empty incremental batch.
dbutils.fs.ls(base_landing_path)

try:
    landing_entries = dbutils.fs.ls(landing_path)
except Exception as exc:
    if not is_missing_path_error(exc):
        raise
    landing_entries = []

# Ignore directories and Spark/OS marker files, then require the configured data format.
# A folder containing only markers or unexpected extensions is not a valid empty batch.
visible_files = [
    entry for entry in landing_entries
    if not entry.isDir()
    and not entry.name.startswith((".", "_"))
]
expected_suffix = ".csv" if normalized_source_file_format == "csv" else ".parquet"
landing_files = [
    entry for entry in visible_files
    if entry.name.lower().endswith(expected_suffix)
]
unexpected_files = [
    entry.name for entry in visible_files
    if not entry.name.lower().endswith(expected_suffix)
]
if unexpected_files:
    raise ValueError(
        f"Landing batch contains files that do not match {normalized_source_file_format or 'parquet'}: "
        f"{sorted(unexpected_files)}. Bronze ingestion was blocked."
    )

if not landing_files:
    stop_for_no_data("No landing data files found for this batch.")


In [0]:
normalized_file_format = normalized_source_file_format
if normalized_file_format in ("", "parquet"):
    landing_df = spark.read.format("parquet").load(landing_path)
elif normalized_file_format == "csv":
    landing_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(landing_path)
else:
    raise ValueError(
        f"Unsupported source_file_format '{source_file_format}'. Expected csv or parquet."
    )

# Treat an empty extract as NO_DATA for incremental loads; never overwrite FULL Silver with zero rows.
if landing_df.limit(1).count() == 0:
    stop_for_no_data("Landing batch contains zero records.")

# Fail fast on schema drift before writing to Bronze. The first accepted batch
# establishes the Bronze schema; later batches must match until a controlled
# schema migration is performed.
bronze_table = f"healthgrid.bronze.{table_name}"

incoming_fields = landing_df.schema.fields
incoming_names = [field.name for field in incoming_fields]
incoming_names_lower = [name.lower() for name in incoming_names]
if len(incoming_names_lower) != len(set(incoming_names_lower)):
    raise ValueError(
        "Incoming source contains column names that collide case-insensitively; "
        "Bronze write was blocked."
    )

reserved_columns = {"landing_timestamp", "insert_timestamp"}
reserved_source_columns = sorted(
    name for name in incoming_names if name.lower() in reserved_columns
)
if reserved_source_columns:
    raise ValueError(
        "Incoming source contains reserved Bronze metadata column(s) "
        f"{reserved_source_columns}; rename the source fields before ingestion."
    )

if spark.catalog.tableExists(bronze_table):
    bronze_df = spark.table(bronze_table)
    bronze_fields = bronze_df.schema.fields
    bronze_names = [field.name for field in bronze_fields]
    bronze_names_lower = [name.lower() for name in bronze_names]
    if len(bronze_names_lower) != len(set(bronze_names_lower)):
        raise ValueError(
            "Existing Bronze table contains column names that collide case-insensitively; "
            "apply a reviewed schema migration before ingestion."
        )

    missing_technical = sorted(reserved_columns - set(bronze_names_lower))
    noncanonical_technical = sorted(
        name for name in bronze_names
        if name.lower() in reserved_columns and name not in reserved_columns
    )
    if missing_technical or noncanonical_technical:
        raise ValueError(
            "Existing Bronze table is missing or has non-canonical batch metadata columns. "
            f"Missing: {missing_technical}; non-canonical: {noncanonical_technical}."
        )

    existing_schema = {
        field.name.lower(): field.dataType.simpleString()
        for field in bronze_fields
        if field.name.lower() not in reserved_columns
    }
    incoming_schema = {
        field.name.lower(): field.dataType.simpleString()
        for field in incoming_fields
    }

    missing_columns = sorted(set(existing_schema) - set(incoming_schema))
    new_columns = sorted(set(incoming_schema) - set(existing_schema))
    changed_types = {
        name: {"bronze": existing_schema[name], "incoming": incoming_schema[name]}
        for name in sorted(set(existing_schema) & set(incoming_schema))
        if existing_schema[name] != incoming_schema[name]
    }
    if missing_columns or new_columns or changed_types:
        raise ValueError(
            "Source schema drift detected; Bronze write was blocked. "
            f"Missing columns: {missing_columns}; new columns: {new_columns}; "
            f"data type changes: {changed_types}. Apply a reviewed schema migration before retrying."
        )

    technical_types = {
        field.name: field.dataType.simpleString()
        for field in bronze_fields
        if field.name in reserved_columns
    }
    expected_technical_types = {
        "landing_timestamp": "string",
        "insert_timestamp": "timestamp",
    }
    if technical_types != expected_technical_types:
        raise TypeError(
            "Existing Bronze technical metadata column types do not match the ingestion contract. "
            f"Expected {expected_technical_types}; found {technical_types}."
        )

    # ADF reruns that reuse the same landing timestamp must not append the same
    # landing batch twice to Bronze. This is a batch replay guard, not a
    # distributed concurrency lock or business-row deduplication mechanism.
    from pyspark.sql import functions as F
    existing_batch = (
        bronze_df
        .filter(F.col("landing_timestamp") == landing_timestamp)
        .limit(1)
        .count()
    )
    if existing_batch:
        raise RuntimeError(
            f"Landing batch '{landing_timestamp}' already exists in {bronze_table}. "
            "Replay is blocked; reconcile the existing batch before retrying."
        )



In [0]:
from pyspark.sql import functions as F

updated_df = landing_df.withColumn("landing_timestamp", F.lit(landing_timestamp)) \
    .withColumn("insert_timestamp", F.current_timestamp())


In [0]:
updated_df.write.mode("append").saveAsTable(f"healthgrid.bronze.{table_name}")

# Emit a stable status for ADF downstream branching.
dbutils.notebook.exit("DATA")
